# XGBoost — Scale Pos Weight + Full Data Retrain

New techniques not used in previous XGBoost notebooks:

1. **`scale_pos_weight`** — corrects class imbalance by upweighting the minority (churn) class
2. **`grow_policy='lossguide'`** — leaf-wise tree growth (like LightGBM), controlled by `max_leaves`
3. **`max_bin=256`** — explicit histogram bin count (was implicit default before)
4. **Full Data Retrain** — after K-Fold finds optimal iterations, retrain a single model on 100% of train data using `best_round × K/(K-1)` rounds
   - K-Fold trains on `(K-1)/K` data → slightly underfit vs full data
   - Scaling compensates: with 10 folds, multiply by `10/9 ≈ 1.111`
   - Produces a stronger single model to blend with the ensemble

Everything else (Optuna with `xgb.cv()`, multi-seed ensemble, feature engineering, tqdm, GPU) carried over from notebook 06.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
from tqdm.auto import tqdm
import warnings
import subprocess
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ─────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')

def has_local_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU  = IS_KAGGLE or has_local_gpu()
DATA_DIR = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU          : {'Enabled ✓' if USE_GPU else 'Disabled — using CPU'}")
print(f"Data dir     : {DATA_DIR}")
print(f"XGBoost ver  : {xgb.__version__}")

# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING   = True
N_TRIALS     = 100
N_CV_FOLDS   = 5      # folds inside each Optuna trial
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10     # folds for the ensemble
TOTAL_MODELS = len(SEEDS) * N_SPLITS

PRECOMPUTED_PARAMS = {
    'learning_rate':     0.015,
    'max_depth':         8,
    'min_child_weight':  5,
    'subsample':         0.80,
    'colsample_bytree':  0.70,
    'colsample_bylevel': 0.80,
    'gamma':             0.10,
    'reg_alpha':         0.05,
    'reg_lambda':        0.50,
    'scale_pos_weight':  2.5,
    'grow_policy':       'lossguide',
    'max_leaves':        64,
}

## 1. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']           = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio']          = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df

print('Feature engineering function defined.')

## 2. Load & Preprocess Data

In [ ]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# ── Compute scale_pos_weight from class distribution ─────────────────────────
n_neg = (train_df[TARGET] == 0).sum()
n_pos = (train_df[TARGET] == 1).sum()
AUTO_SCALE_POS_WEIGHT = n_neg / n_pos
print(f'\nClass distribution — Negative: {n_neg:,}  Positive: {n_pos:,}')
print(f'Auto scale_pos_weight = {n_neg}/{n_pos} = {AUTO_SCALE_POS_WEIGHT:.4f}')
print('(Optuna will search around this value)')

train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features     = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
cat_features = []

print('\nLabel encoding categorical features...')
for col in tqdm(features, desc='Encoding'):
    if df[col].dtype == 'object':
        cat_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

num_features = [c for c in features if c not in cat_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

print('\nEngineering features...')
X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

for col in cat_features:
    X[col]      = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f'\nTrain : {X.shape}  |  Test : {X_test.shape}')

## 3. Optuna Tuning

New parameters in the search space vs notebook 06:
- **`scale_pos_weight`** — searched around the auto-computed class ratio
- **`grow_policy`** — chooses between `depthwise` and `lossguide` (leaf-wise)
- **`max_leaves`** — only active when `grow_policy='lossguide'`
- **`max_bin`** — explicitly tuned (was left at default 256 before)

In [ ]:
base_params = {
    'objective':   'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity':   0,
    'seed':        42,
}
if USE_GPU:
    base_params['device'] = 'cuda'
    print('GPU params injected: device=cuda')

dtrain_full = xgb.DMatrix(X, label=y, enable_categorical=True)
dtest       = xgb.DMatrix(X_test, enable_categorical=True)

if RUN_TUNING:
    print(f'\nStarting Optuna tuning: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV each')

    pbar = tqdm(total=N_TRIALS, desc='Optuna Trials', unit='trial')
    best_value_so_far = [0.0]

    def objective(trial):
        grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])

        params = {
            **base_params,
            'learning_rate':     trial.suggest_float('learning_rate',     0.005, 0.1,   log=True),
            'max_depth':         trial.suggest_int(  'max_depth',         3,     10),
            'min_child_weight':  trial.suggest_int(  'min_child_weight',  1,     20),
            'subsample':         trial.suggest_float('subsample',         0.5,   1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',  0.3,   1.0),
            'gamma':             trial.suggest_float('gamma',             1e-8,  5.0,   log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha',         1e-8,  10.0,  log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',        1e-8,  10.0,  log=True),
            # ── New parameters ────────────────────────────────────────────────
            'scale_pos_weight':  trial.suggest_float('scale_pos_weight',
                                                     max(1.0, AUTO_SCALE_POS_WEIGHT * 0.5),
                                                     AUTO_SCALE_POS_WEIGHT * 2.0),
            'grow_policy':       grow_policy,
            'max_leaves':        trial.suggest_int('max_leaves', 16, 128) if grow_policy == 'lossguide' else 0,
            'max_bin':           trial.suggest_int('max_bin', 128, 512),
        }

        cv_result = xgb.cv(
            params,
            dtrain_full,
            num_boost_round      = 2000,
            nfold                = N_CV_FOLDS,
            stratified           = True,
            metrics              = 'auc',
            early_stopping_rounds= 50,
            verbose_eval         = False,
            seed                 = 42,
        )

        score     = cv_result['test-auc-mean'].max()
        best_iter = int(cv_result['test-auc-mean'].idxmax())
        trial.set_user_attr('best_iteration', best_iter)

        if score > best_value_so_far[0]:
            best_value_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_value_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial = study.best_trial
    best_round = best_trial.user_attrs.get('best_iteration', 500)

    print(f"\n{'='*55}")
    print(f"Best CV AUC    : {best_trial.value:.5f}")
    print(f"Best round     : {best_round}")
    for k, v in best_trial.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    best_params = {**base_params, **best_trial.params}
    study.trials_dataframe().to_csv('xgb_scaledpos_tuning_results.csv', index=False)
    print('\nAll trial results saved → xgb_scaledpos_tuning_results.csv')

else:
    print('Skipping tuning — using PRECOMPUTED_PARAMS.')
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_round  = 500
    for k, v in PRECOMPUTED_PARAMS.items():
        print(f'  {k:25s}: {v}')

## 4. Multi-Seed Ensemble

In [ ]:
print(f'Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n')

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc='Seeds', position=0)

for seed in outer_bar:
    outer_bar.set_description(f'Seed {seed}')
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    inner_bar = tqdm(
        enumerate(skf.split(X, y)),
        total=N_SPLITS, desc='  Folds', position=1, leave=False
    )

    for fold, (train_idx, val_idx) in inner_bar:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        dtrain_fold = xgb.DMatrix(X_tr,  label=y_tr,  enable_categorical=True)
        dval_fold   = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

        model = xgb.train(
            {**best_params, 'seed': seed},
            dtrain_fold,
            num_boost_round      = best_round + 200,
            evals                = [(dval_fold, 'val')],
            early_stopping_rounds= 150,
            verbose_eval         = False,
        )

        itr        = (0, model.best_iteration + 1)
        val_preds  = model.predict(dval_fold, iteration_range=itr)
        test_preds = model.predict(dtest,     iteration_range=itr)

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]        = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum          += test_preds

        inner_bar.set_postfix({'fold_auc': f'{fold_auc:.5f}'})
        del model, dtrain_fold, dval_fold
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({'seed_oof_auc': f'{seed_auc:.5f}'})

global_oof      = global_oof_sum / len(SEEDS)
ensemble_auc    = roc_auc_score(y, global_oof)
ensemble_preds  = test_preds_sum / TOTAL_MODELS

print(f"\n{'='*55}")
print(f"Fold AUC — mean : {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Ensemble OOF AUC: {ensemble_auc:.5f}")
print(f"{'='*55}")

## 5. Full Data Retrain (Single Model on 100% Train)

K-Fold trains on `(K-1)/K` of data, so `best_round` slightly underestimates the optimal iterations for the full dataset.

**Formula:** `full_rounds = int(best_round × N_SPLITS / (N_SPLITS - 1))`

With 10 folds: `full_rounds = best_round × 10/9 ≈ best_round × 1.111`

No early stopping — we already know the right number of rounds from CV.

In [ ]:
full_rounds = int(best_round * N_SPLITS / (N_SPLITS - 1))
print(f'best_round (from CV)    : {best_round}')
print(f'scale factor            : {N_SPLITS}/{N_SPLITS-1} = {N_SPLITS/(N_SPLITS-1):.4f}')
print(f'full_rounds (100% data) : {full_rounds}')

print('\nRetraining single model on 100% train data...')

full_model = xgb.train(
    {**best_params, 'seed': 42},
    dtrain_full,                 # entire training set
    num_boost_round = full_rounds,
    verbose_eval    = False,
    # No eval_set, no early_stopping — rounds are already calibrated
)

full_preds = full_model.predict(dtest)
print(f'Full data model prediction range: [{full_preds.min():.4f}, {full_preds.max():.4f}]')

# ── Blend ensemble + full-data model ─────────────────────────────────────────
# Simple 50/50 blend — often better than either alone
FULL_MODEL_WEIGHT = 0.5   # adjust: higher = more weight on full-data model
blended_preds = (1 - FULL_MODEL_WEIGHT) * ensemble_preds + FULL_MODEL_WEIGHT * full_preds

print(f'\nBlend weight — Ensemble: {1-FULL_MODEL_WEIGHT:.1f}  Full-data: {FULL_MODEL_WEIGHT:.1f}')

## 6. Save Submissions

In [ ]:
# Ensemble submission
sub_ensemble = sub.copy()
sub_ensemble[TARGET] = ensemble_preds
sub_ensemble.to_csv('submission_xgb_scaledpos_ensemble.csv', index=False)
print('Saved → submission_xgb_scaledpos_ensemble.csv')

# Full data retrain submission
sub_full = sub.copy()
sub_full[TARGET] = full_preds
sub_full.to_csv('submission_xgb_scaledpos_fulldata.csv', index=False)
print('Saved → submission_xgb_scaledpos_fulldata.csv')

# Ensemble + full data blend
sub_blend = sub.copy()
sub_blend[TARGET] = blended_preds
sub_blend.to_csv('submission_xgb_scaledpos_blend.csv', index=False)
print('Saved → submission_xgb_scaledpos_blend.csv')

print(f'\nEnsemble OOF AUC : {ensemble_auc:.5f}')
display(sub_blend.head())